In [2]:
import mlflow
import pandas as pd
import nltk; 
nltk.download("wordnet")
nltk.download("stopwords")
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv')

In [4]:
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer= WordNetLemmatizer()
    text= text.split()
    text= [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)
def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)
def normalize_text(df):
    """Normalize the text data."""
    try:
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [5]:
df= normalize_text(df)
df

,tweet_id,sentiment,content
0,1956967341,empty,tiffanylue know listenin bad habit earlier sta...
1,1956967666,sadness,layin n bed headache ughhhh waitin call
2,1956967696,sadness,funeral ceremony gloomy friday
3,1956967789,enthusiasm,want hang friend soon
4,1956968416,neutral,dannycastillo want trade someone houston ticke...
...,...,...,...
39995,1753918954,neutral,johnlloydtaylor
39996,1753919001,love,happy mother day love
39997,1753919005,love,happy mother s day mommy there woman man long ...
39998,1753919043,happiness,niariley wassup beautiful follow me peep new h...


In [6]:
df.drop('tweet_id', axis=1, inplace=True)

In [7]:
x = df['sentiment'].isin(['happiness','sadness'])
df = df[x]

In [8]:
df['sentiment']= df['sentiment'].replace({'happiness':1, 'sadness':0})  
df.head(5)

C:\Users\hp\AppData\Local\Temp\ipykernel_33972\3291795485.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment']= df['sentiment'].replace({'happiness':1, 'sadness':0})
C:\Users\hp\AppData\Local\Temp\ipykernel_33972\3291795485.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sentiment']= df['sentiment'].replace({'happiness':1, 'sadness':0})


,sentiment,content
1,0,layin n bed headache ughhhh waitin call
2,0,funeral ceremony gloomy friday
6,0,sleep im not thinking old friend want married ...
8,0,charviray charlene love miss
9,0,kelcouch sorry least friday


In [9]:
vectorizer = CountVectorizer(max_features=1000)
X= vectorizer.fit_transform(df['content'])
Y= df['sentiment']
df.head(5)

,sentiment,content
1,0,layin n bed headache ughhhh waitin call
2,0,funeral ceremony gloomy friday
6,0,sleep im not thinking old friend want married ...
8,0,charviray charlene love miss
9,0,kelcouch sorry least friday


In [10]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size=0.2, random_state=42)    

In [11]:
import dagshub
dagshub.init(repo_owner='pandeyshivam8765', repo_name='MLops_mini_project', mlflow=True)
mlflow.set_tracking_uri('https://dagshub.com/pandeyshivam8765/MLops_mini_project.mlflow')
mlflow.set_experiment('logistic_regression_baseline')

Accessing as pandeyshivam8765

Initialized MLflow to track repo "pandeyshivam8765/MLops_mini_project"

Repository pandeyshivam8765/MLops_mini_project initialized!

<Experiment: artifact_location='mlflow-artifacts:/89c4638556664c88b93acf7b6e348b03', creation_time=1774269585014, experiment_id='1', last_update_time=1774269585014, lifecycle_stage='active', name='logistic_regression_baseline', tags={}, workspace='default'>

In [12]:
with mlflow.start_run():
    
    # Log preprocessing parameters
    mlflow.log_param("vectorizer", "Bag of Words")
    mlflow.log_param("num_features", 1000)
    mlflow.log_param("test_size", 0.2)
    
    # Model building and training
    model = LogisticRegression()
    model.fit(X_train, Y_train)
    
    # Log model parameters
    mlflow.log_param("model", "Logistic Regression")
    
    # Model evaluation
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(Y_test, y_pred)
    precision = precision_score(Y_test, y_pred)
    recall = recall_score(Y_test, y_pred)
    f1 = f1_score(Y_test, y_pred)

    # Log evaluation metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

🏃 View run bustling-cow-747 at: https://dagshub.com/pandeyshivam8765/MLops_mini_project.mlflow/#/experiments/1/runs/8db2f56b6eba4c319c6350b56e8b4e71
🧪 View experiment at: https://dagshub.com/pandeyshivam8765/MLops_mini_project.mlflow/#/experiments/1
